# YOLO 自定义数据集训练教程
本Notebook演示完整的YOLO训练流程：从环境配置到模型训练和推理。

适用场景：轨道检测、设备状态检测等垂直领域应用

## 1. 环境配置

In [ ]:
# 安装依赖
!pip install ultralytics opencv-python matplotlib -q

In [ ]:
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
from IPython.display import Image, display
import os

# 检查GPU
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 准备数据集

### 数据集目录结构
```
datasets/custom_dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

### 标签格式
每个标签文件(.txt)包含：
```
<class_id> <x_center> <y_center> <width> <height>
```
坐标和大小均为归一化值 (0-1)。

In [ ]:
# 创建数据集目录结构
import os
from pathlib import Path

dataset_path = Path('datasets/custom_dataset')

dirs = [
    dataset_path / 'images' / 'train',
    dataset_path / 'images' / 'val',
    dataset_path / 'images' / 'test',
    dataset_path / 'labels' / 'train',
    dataset_path / 'labels' / 'val',
    dataset_path / 'labels' / 'test',
]

for dir_path in dirs:
    dir_path.mkdir(parents=True, exist_ok=True)

print("✅ 数据集目录结构已创建")
print(f"路径: {dataset_path.absolute()}")

## 3. 创建数据集配置文件

In [ ]:
import yaml

# 数据集配置
data_config = {
    'path': str(dataset_path.absolute()),  # 数据集根路径
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    
    'nc': 3,  # 类别数
    'names': {  # 类别名称
        0: 'track',
        1: 'defect',
        2: 'fastener'
    }
}

# 保存配置文件
config_path = 'data.yaml'
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, allow_unicode=True)

print(f"✅ 配置文件已保存: {config_path}")
print(f"\n配置内容:")
print(yaml.dump(data_config, allow_unicode=True))

## 4. 下载和加载预训练模型

In [ ]:
# 选择模型版本
# - yolov8n.pt: nano (最快，参数最少)
# - yolov8s.pt: small
# - yolov8m.pt: medium (推荐)
# - yolov8l.pt: large
# - yolov8x.pt: xlarge (最准，参数最多)

model_name = 'yolov8n.pt'  # 推荐从小模型开始

print(f"📥 加载模型: {model_name}")
model = YOLO(model_name)  # 会自动下载

print(f"✅ 模型加载成功")
print(f"参数量: {sum(p.numel() for p in model.model.parameters()):,}")

## 5. 开始训练

⚠️ **注意**: 确保你已将数据放入上面创建的目录中！

In [ ]:
# 训练参数
EPOCHS = 100         # 训练轮数
BATCH_SIZE = 16      # 批大小 (根据显存调整)
IMG_SIZE = 640       # 图像大小
PATIENCE = 50        # 早停耐心值

print(f"🏋️  开始训练...")
print(f"训练参数: epochs={EPOCHS}, batch={BATCH_SIZE}, imgsz={IMG_SIZE}")
print("="*60)

# 开始训练
results = model.train(
    data=config_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=0,  # 使用GPU 0，如果没有GPU则自动使用CPU
    project='runs/train',
    name='exp',
    exist_ok=True,
    
    # 数据增强
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    
    # 性能优化
    amp=True,  # 自动混合精度
    plots=True,  # 生成训练图表
    save=True,
    verbose=True
)

print("\n" + "="*60)
print("✅ 训练完成!")

## 6. 查看训练结果

In [ ]:
# 显示训练曲线
from IPython.display import Image

result_dir = 'runs/train/exp'

# 训练结果图
print("📊 训练曲线:")
display(Image(filename=f'{result_dir}/results.png'))

# 混淆矩阵
print("\n📊 混淆矩阵:")
display(Image(filename=f'{result_dir}/confusion_matrix.png'))

## 7. 模型验证

In [ ]:
# 加载最佳模型
best_model = YOLO('runs/train/exp/weights/best.pt')

# 验证
metrics = best_model.val(data=config_path)

print("\n📊 验证结果:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 8. 推理测试

In [ ]:
# 对单张图像进行推理
test_image = 'path/to/your/test_image.jpg'  # 替换为你的测试图像路径

# 推理
results = best_model.predict(
    source=test_image,
    conf=0.25,  # 置信度阈值
    save=True,
    project='runs/predict',
    name='exp',
    exist_ok=True
)

# 显示结果
print("\n🔍 检测结果:")
for result in results:
    boxes = result.boxes
    print(f"检测到 {len(boxes)} 个对象:")
    
    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        cls_name = result.names[cls_id]
        print(f"  - {cls_name}: {conf:.2f}")

# 显示标注后的图像
display(Image(filename='runs/predict/exp/test_image.jpg'))

## 9. 批量推理

In [ ]:
# 对整个目录进行推理
test_dir = 'datasets/custom_dataset/images/test'  # 测试集目录

results = best_model.predict(
    source=test_dir,
    conf=0.25,
    save=True,
    project='runs/predict',
    name='batch_exp',
    exist_ok=True
)

print(f"✅ 批量推理完成，结果保存在: runs/predict/batch_exp")

## 10. 导出模型

In [ ]:
# 导出为ONNX格式（用于部署）
export_path = best_model.export(format='onnx')
print(f"✅ 模型已导出: {export_path}")

# 其他可用格式:
# - 'torchscript': PyTorch
# - 'onnx': ONNX
# - 'engine': TensorRT
# - 'coreml': CoreML (iOS)
# - 'tflite': TensorFlow Lite (移动端)

## 11. 性能基准测试

In [ ]:
import time

test_image = 'path/to/test_image.jpg'
iterations = 100

# 预热
for _ in range(10):
    _ = best_model.predict(test_image, verbose=False)

# 基准测试
start_time = time.time()
for _ in range(iterations):
    _ = best_model.predict(test_image, verbose=False)
elapsed = time.time() - start_time

avg_time = elapsed / iterations
fps = iterations / elapsed

print(f"\n⚡ 性能基准测试:")
print(f"  平均推理时间: {avg_time*1000:.2f} ms")
print(f"  吞吐量 (FPS): {fps:.1f}")

## 总结

🎉 你已经完成了YOLO自定义数据集的完整训练流程！

### 下一步:
1. ✅ 收集更多数据以提高精度
2. ✅ 尝试不同的模型大小 (yolov8m, yolov8l)
3. ✅ 调整超参数优化性能
4. ✅ 部署到生产环境

### 文件位置:
- 最佳模型: `runs/train/exp/weights/best.pt`
- 训练结果: `runs/train/exp/`
- 推理结果: `runs/predict/`